In [1]:
import sys
sys.path.append("../src")
import pandas as pd
from utils.utils import eval, generate, to_csv
from utils.models import get_models
from collections import Counter

In [2]:
models = get_models()
df = pd.read_excel("../data/ai_lab_1.xlsx")
gr_truth = pd.read_csv("../data/ground_truth.csv")

In [ ]:
def make_cot_prompt_hidde(property_id: str, input: str) -> str:
    pid = str(property_id)
    return f"""
You are a deterministic JSON extraction engine.

Think step-by-step internally before answering.
Do NOT output your reasoning.

Return EXACTLY one JSON object and NOTHING else.

Output format:
{{ "{pid}": number | null }}

Task:
Extract the total land plot area.

Allowed units:
- m2, sqm, square meters
- sq ft, sqft, ft2
- hectares (ha)

Conversions:
- 1 hectare = 10000 square meters
- 1 square foot = 0.092903 square meters

Rounding:
Round to exactly 2 decimals.

Internal reasoning steps (do not output):
1. Scan the text for numbers associated with land/plot area.
2. Ignore building area, living area, perimeter, distances, and heights.
3. Identify the unit (m2, sqft, ha).
4. Normalize the number (remove commas if needed).
5. Convert to square meters if required.
6. Round to exactly 2 decimals.
7. Output JSON.

If no valid land plot area exists:
{{ "{pid}": null }}


<text>
{input}
</text>
""".strip()

In [4]:
# generate multiple outputs using the same input, and get the most frequent
def self_consistency_generate(model:str, prompt: str):
    results = []
    for _ in range(4):
        output = generate(model=model, prompt=prompt)
        results.append(output)

    filtered = [p for p in results if p is not None]
    return Counter(filtered).most_common(1)[0][0]

In [5]:
for model in models:
    print(f"start for {model.short_name}")
    res = eval(model=model.model_name, df=df, gr_truth=gr_truth, make_prompt_fn=make_cot_prompt_hidde, generate_fn=self_consistency_generate)
    to_csv(model.short_name, "self_consistency_prompt.csv", res)
    print(f"done for {model.short_name}")

print("done")

start for llama


100%|██████████| 117/117 [02:48<00:00,  1.44s/it]


done for llama
start for gemma


100%|██████████| 117/117 [08:08<00:00,  4.18s/it]

done for gemma
done
